# 07 — Risk appetite statement + cascaded limit framework

**Plain English:** The metrics in notebooks 02–06 tell us *what the book is
doing*; this notebook says *what we will tolerate*. We read a small **risk
appetite table** from `config/risk_appetite.yaml` — an amber and a red limit, an
owner, a breach action and a review cycle for each metric — then score the
current book **green / amber / red (RAG)** against those limits. Monitoring
without limits is just reporting (APS 220 para 20).

**One-line terms**
- **Risk appetite** — how much risk the Board is willing to run; expressed as limits.
- **Amber / Red limit** — early-warning level / hard limit; a red breach escalates to the Board.
- **RAG status** — green within appetite · amber approaching the limit · red breached.
- **Leading vs lagging** — leading indicators (SICR share, roll rates) move *before* defaults; lagging ones (NPL) confirm after (APG 220 para 66).
- **Cascade** — appetite flows Board → portfolio → segment → facility (APS 220 para 35).

> Illustrative demo thresholds — **not a regulatory submission**. Levels are set
> to plausible mortgage values, not fitted to this crisis+calm sample.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
pd.set_option("display.width", 120); pd.set_option("display.max_columns", 30)
import monitor as m
print("monitor library loaded — vintages:", m.VINTAGES)
panel = pd.read_parquet(m.PROC_DIR / "panel.parquet")
cfg = m.load_appetite()
print("appetite metrics:", list(cfg["metrics"]))

monitor library loaded — vintages: ['2007', '2008', '2015']


appetite metrics: ['npl_ratio', 'stage2_share', 'loss_rate', 'provision_coverage', 'high_lvr_share', 'current_high_lvr_share', 'top_state_share', 'geo_hhi', 'roll_30_60', 'roll_current_30']


In [2]:
# Current vs prior reporting month, all appetite metrics as-of each --------
periods = sorted(int(p) for p in panel["period"].unique())
this_period = periods[-1]
def _prev(p):
    y, mn = divmod(p, 100)
    return (y - 1) * 100 + 12 if mn == 1 else p - 1
last_period = _prev(this_period) if _prev(this_period) in set(periods) else periods[-2]

cov = cfg.get("ecl", {}).get("coverage_rates")
this_vals = m.portfolio_metrics_asof(panel, this_period, coverage=cov)
last_vals = m.portfolio_metrics_asof(panel, last_period, coverage=cov)
appetite = m.evaluate_appetite(cfg, this_vals, last_vals)
appetite.to_csv(m.OUT_TABLES / "07_appetite_status.csv", index=False)
print(f"this period {this_period} vs last period {last_period}")
appetite[["metric", "type", "last_period", "this_period", "amber", "red (limit)", "RAG"]]

this period 202509 vs last period 202508


,metric,type,last_period,this_period,amber,red (limit),RAG
0,NPL ratio (Stage 3 / 90+ share of EAD),lagging,0.90,0.86,2.0,4.0,GREEN
1,Stage 2 share of EAD (SICR watch book),leading,2.01,1.90,5.0,8.0,GREEN
2,Expected-loss rate (bps of EAD),lagging,40.66,39.30,50.0,75.0,GREEN
3,NPL provision coverage (ECL / Stage 3 exposure...,lagging,45.29,45.75,35.0,25.0,GREEN
4,High-LVR book share (original LVR > 90% of EAD),leading,12.96,13.00,15.0,25.0,GREEN
5,Current/indexed high-LVR share (current LVR > ...,leading,0.00,0.00,10.0,18.0,GREEN
6,Geographic concentration (top-state exposure s...,lagging,17.30,17.27,20.0,30.0,GREEN
7,"Geographic concentration (state HHI, 0-10,000)",lagging,569.15,568.35,1500.0,2500.0,GREEN
8,30->60 roll rate (trailing 12m),leading,15.49,15.73,20.0,30.0,GREEN
9,"New-delinquency roll (Current->30, trailing 12m)",leading,0.69,0.69,1.0,2.0,GREEN


In [3]:
# Leading-indicator TREND over time (not just the pooled matrix) ----------
# APG 220 para 66: a prudent ADI uses forward-looking indicators. We track the
# trailing-12m roll rates and the SICR (Current->Stage 2) migration rate at each
# year-end, so the trend is visible, not just a single pooled number.
def sicr_rate(trans, end_period, months=12):
    end = m._period_ord(end_period)
    sub = trans[(trans["bucket"] == "Current") & trans["next_stage"].notna()].copy()
    o = m._period_ord(sub["period"])
    win = sub[(o <= end) & (o > end - months)]
    return float(100 * (win["next_stage"] == "Stage 2").mean()) if len(win) else np.nan

anchors = [p for p in periods if p % 100 == 12][-5:] or periods[-5:]
if this_period not in anchors:
    anchors = anchors + [this_period]
trend = pd.DataFrame([{
    "as_of": p,
    "roll_current_30 (leading)": round(m.roll_window(panel, p)["roll_current_30"], 3),
    "roll_30_60 (leading)": round(m.roll_window(panel, p)["roll_30_60"], 2),
    "sicr_current_to_stage2 (leading)": round(sicr_rate(panel, p), 3),
} for p in anchors])
trend.to_csv(m.OUT_TABLES / "07_leading_trends.csv", index=False)
trend

,as_of,roll_current_30 (leading),roll_30_60 (leading),sicr_current_to_stage2 (leading)
0,202012,1.060,35.07,1.063
1,202112,0.624,17.66,0.625
2,202212,0.698,17.95,0.700
3,202312,0.646,15.41,0.649
4,202412,0.717,14.55,0.719
5,202509,0.686,15.73,0.689


In [4]:
# Stress -> limits: graded downturn scenarios (MON-7) -------------------
# APS 220 paras 73/76. Per-metric multipliers (config) reflect that watch/roll FLOW
# metrics move more than the NPL STOCK in a downturn — grounded in this repo's
# 2007-vs-2015 vintage ratios. Two scenarios give the Board a graded picture.
stress_frames = []
for scen in ["moderate", "severe"]:
    s = m.stress_table(this_vals, cfg, scen)
    s.insert(0, "scenario", cfg["stress"]["scenarios"][scen]["label"])
    stress_frames.append(s)
stress = pd.concat(stress_frames, ignore_index=True)
stress.to_csv(m.OUT_TABLES / "07_stress_vs_limits.csv", index=False)
print(cfg["stress"]["note"])
stress

Per-metric downturn multipliers grounded in this repo's 2007-vs-2015 vintage ratios (2007 cumulative default ~4x 2015 at 72 months on book).


,scenario,metric,current,multiplier,stressed,red (limit),RAG current,RAG stressed
0,Moderate downturn,NPL ratio (Stage 3 / 90+ share of EAD),0.86,1.8,1.55,4.0,GREEN,GREEN
1,Moderate downturn,Stage 2 share of EAD (SICR watch book),1.90,2.0,3.81,8.0,GREEN,GREEN
2,Moderate downturn,Expected-loss rate (bps of EAD),39.30,1.8,70.75,75.0,GREEN,AMBER
3,Moderate downturn,30->60 roll rate (trailing 12m),15.73,1.8,28.31,30.0,GREEN,AMBER
4,Moderate downturn,"New-delinquency roll (Current->30, trailing 12m)",0.69,1.8,1.24,2.0,GREEN,AMBER
5,Severe downturn (GFC-like),NPL ratio (Stage 3 / 90+ share of EAD),0.86,3.0,2.58,4.0,GREEN,AMBER
6,Severe downturn (GFC-like),Stage 2 share of EAD (SICR watch book),1.90,3.5,6.67,8.0,GREEN,AMBER
7,Severe downturn (GFC-like),Expected-loss rate (bps of EAD),39.30,3.0,117.91,75.0,GREEN,RED
8,Severe downturn (GFC-like),30->60 roll rate (trailing 12m),15.73,3.0,47.19,30.0,GREEN,RED
9,Severe downturn (GFC-like),"New-delinquency roll (Current->30, trailing 12m)",0.69,3.0,2.06,2.0,GREEN,RED


In [5]:
# Limit utilisation / headroom (monthly analog of DAILY limit monitoring) -
# APS 113 Att.D EAD para 6 / Basel CRE36.92 require DAILY facility/limit-excess
# monitoring; the true daily layer is described in docs/governance.md. Here we show
# how much of each limit is used and the headroom left at the reporting date.
util = m.limit_utilisation(cfg, this_vals)
util.to_csv(m.OUT_TABLES / "07_limit_utilisation.csv", index=False)
util

,metric,this_period,limit (red),utilisation_vs_limit_pct,headroom_to_limit,RAG
0,NPL ratio (Stage 3 / 90+ share of EAD),0.86,4.0,21.5,3.14,GREEN
1,Stage 2 share of EAD (SICR watch book),1.90,8.0,23.8,6.10,GREEN
2,Expected-loss rate (bps of EAD),39.30,75.0,52.4,35.70,GREEN
3,NPL provision coverage (ECL / Stage 3 exposure...,45.75,25.0,54.6,20.75,GREEN
4,High-LVR book share (original LVR > 90% of EAD),13.00,25.0,52.0,12.00,GREEN
5,Current/indexed high-LVR share (current LVR > ...,0.00,18.0,0.0,18.00,GREEN
6,Geographic concentration (top-state exposure s...,17.27,30.0,57.6,12.73,GREEN
7,"Geographic concentration (state HHI, 0-10,000)",568.35,2500.0,22.7,1931.65,GREEN
8,30->60 roll rate (trailing 12m),15.73,30.0,52.4,14.27,GREEN
9,"New-delinquency roll (Current->30, trailing 12m)",0.69,2.0,34.3,1.31,GREEN


In [6]:
# IFRS 9 ECL provision & coverage ratio (APG 220 para 67(b)) ------------
# Provision coverage is a required forward-looking indicator. Stage exposures x the
# (illustrative) config coverage rates -> ECL, the expected-loss rate (bps) and the
# NPL coverage ratio. These also feed the loss_rate / provision_coverage RAG metrics.
book = m.book_asof(panel, this_period)
ecl = m.ecl_table(book, cov)
ecl.to_csv(m.OUT_TABLES / "07_ecl_provisions.csv", index=False)
ecl

,EAD ($),Stage 2 exposure ($),Stage 3 / NPL exposure ($),ECL provision ($),EL rate (bps of EAD),NPL coverage (%)
0,1.735203e+09,33046094.0,14906036.0,6819983.0,39.3,45.8
